<a href="https://colab.research.google.com/github/Julian27R/Aprendizaje_Maquina/blob/main/Dashboard_PFTAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

#**Sistema de Detección de Fallos en Rodamientos (CWRU) con Modelos MLP, SVC y Random Forest vía App Web**

###**Por:**
*   **Julián Felipe Gutiérrez Ramírez**
*   **Samuel Marulanda Salazar**
*   **Felipe Idárraga Quintero**

**Fecha de presentación 20/07/2025**

---

#**1. Descarga de Modelos Entrenados (Random Forest, SVC, MLP)**

In [27]:
import gdown

# Random Forest
#https://drive.google.com/file/d/12g8ebDlU16Ui0wXaOKnHdO8uBbEGJpQ7/view?usp=sharing
gdown.download(id="12g8ebDlU16Ui0wXaOKnHdO8uBbEGJpQ7", output="rf_pipeline.pkl", quiet=False)

# SVC
#https://drive.google.com/file/d/1SSakLrL1Funlq61eXTamukI-aXMJZwD-/view?usp=sharing
gdown.download(id="1SSakLrL1Funlq61eXTamukI-aXMJZwD-", output="svc_pipeline.pkl", quiet=False)

# MLP
#https://drive.google.com/file/d/1XKAe4HZ-_AdyeFEHnsDZVPjw5zzP4ZGJ/view?usp=sharing
gdown.download(id="1XKAe4HZ-_AdyeFEHnsDZVPjw5zzP4ZGJ", output="MLP_model_state_dict.pth", quiet=False)

Downloading...
From: https://drive.google.com/uc?id=12g8ebDlU16Ui0wXaOKnHdO8uBbEGJpQ7
To: /content/rf_pipeline.pkl
100%|██████████| 1.18M/1.18M [00:00<00:00, 137MB/s]
Downloading...
From: https://drive.google.com/uc?id=1SSakLrL1Funlq61eXTamukI-aXMJZwD-
To: /content/svc_pipeline.pkl
100%|██████████| 75.8k/75.8k [00:00<00:00, 61.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1XKAe4HZ-_AdyeFEHnsDZVPjw5zzP4ZGJ
To: /content/MLP_model_state_dict.pth
100%|██████████| 49.4k/49.4k [00:00<00:00, 33.8MB/s]


'MLP_model_state_dict.pth'

#**2.  Instalación de Dependencias para la App Web (Streamlit y Pyngrok)**

In [2]:
!pip install streamlit pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 6.0 MB/s eta 0:00:00


#**3. Definición de app.py: Interfaz Streamlit para Clasificación de Fallos (MLP, SVC, RF)**

In [30]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import torch
from pathlib import Path
from torch import nn
import numpy as np

#Arquitectura MLP (vector 9-dim → clase)
# ────────────────────────────────────────────────────────────────
class MLPClassifier(nn.Module):
    def __init__(self, n_classes: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(9, 128), nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),
            nn.Linear(64, n_classes)
        )
    def forward(self, x):            # x: [B, 9]
        return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# === Diccionario de clases ===
clases = {
    0: 'Ball_007_1', 1: 'Ball_014_1', 2: 'Ball_021_1',
    3: 'IR_007_1',   4: 'IR_014_1',   5: 'IR_021_1',
    6: 'Normal_1', 7: 'OR_007_6_1', 8: 'OR_014_6_1',
    9: 'OR_021_6_1'
}
clases_inv = {v: k for k, v in clases.items()}

# === Título e interfaz ===
st.title("🛠️ Clasificación de Fallos en Rodamientos (CWRU)")
model_choice = st.selectbox("Selecciona el modelo para predecir:", ["Random Forest", "SVC", "MLP"])

# === Inputs del usuario (hasta 4 decimales) ===
inputs = {
    "max":      st.number_input("max",      0.0,   1000.0, 0.0,  step=0.0001, format="%.4f"),
    "min":      st.number_input("min",   -1000.0,     0.0, 0.0, step=0.0001, format="%.4f"),
    "mean":     st.number_input("mean",   -500.0,   500.0,   0.0,  step=0.0001, format="%.4f"),
    "sd":       st.number_input("sd",        0.0,   100.0,  0.0,  step=0.0001, format="%.4f"),
    "rms":      st.number_input("rms",       0.0,   100.0,  0.0,  step=0.0001, format="%.4f"),
    "skewness": st.number_input("skewness",  -5.0,     5.0,   0.0, step=0.0001, format="%.4f"),
    "kurtosis": st.number_input("kurtosis",  -5.0,    40.0,   0.0, step=0.0001, format="%.4f"),
    "crest":    st.number_input("crest",      -3.0,    10.0,   0.0, step=0.0001, format="%.4f"),
    "form":     st.number_input("form",       0.0,    500.0,   0.0, step=0.0001, format="%.4f"),
}

# === Botón de predicción ===
if st.button("🔍 Predecir clase"):
    X_user = pd.DataFrame([inputs])

    if model_choice == "Random Forest":
        modelo = joblib.load("rf_pipeline.pkl")
        pred   = modelo.predict(X_user)[0]
        proba  = modelo.predict_proba(X_user)[0]
        labels = modelo.classes_

    elif model_choice == "SVC":
        modelo = joblib.load("svc_pipeline.pkl")
        pred   = modelo.predict(X_user)[0]
        proba  = modelo.predict_proba(X_user)[0]
        labels = modelo.classes_

    # --- MLP ---
    else:
        # 1. Cargar modelo entrenado
        modelo = MLPClassifier(10).to(device)
        modelo.load_state_dict(torch.load("MLP_model_state_dict.pth",
                                          map_location=device))
        modelo.eval()

        # 3. Tensor y forward
        x_tensor = torch.tensor(X_user.values, dtype=torch.float32, device=device)

        with torch.no_grad():
            logits = modelo(x_tensor)
            proba  = torch.softmax(logits, dim=1).cpu().numpy()[0]
            pred   = int(proba.argmax())

        labels = [clases[i] for i in range(len(proba))]

    # Mostrar resultado
    pred_idx = clases_inv[pred] if isinstance(pred, str) else pred
    st.success(f"🔎 Resultado: **{clases[pred_idx]}**")

    # Mostrar probabilidades
    st.subheader("📊 Probabilidades por clase")
    st.bar_chart(pd.Series(proba, index=labels))


Overwriting app.py


#**4. Generación del Archivo requirements.txt con Dependencias del Proyecto**

In [31]:
%%writefile requirements.txt
streamlit>=1.28
scikit-learn>=1.3
pandas>=2.0
joblib>=1.3
torch>=2.0


Overwriting requirements.txt


#**5. Despliegue del Dashboard con Streamlit y Pyngrok**

In [6]:
from pyngrok import ngrok

!ngrok config add-authtoken 2xbQ2LIhhqTUCyRMtyHaqy9cZym_inbfFpBVWsG79Gzdy7EA

public_url = ngrok.connect("http://localhost:8501")
print(f"Tu dashboard está disponible en: {public_url}")

!streamlit run app.py &>/content/logs.txt &

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Tu dashboard está disponible en: NgrokTunnel: "https://b860c44ce52f.ngrok-free.app" -> "http://localhost:8501"
